# Re-score all fire-progression seeds on the full 24-fire test set

Inference only, no training. About 15-20 minutes.

Your five FP training runs each evaluated only the 16-fire subset. Reviewer 3
asked for results on the complete official test set, so this re-scores every
checkpoint on all 24 fires and reports each exclusion policy side by side.

Efficient by design: each window is loaded from disk **once** and passed through
all five models, rather than reloading the data five times.

It also sweeps the threshold on test for each seed. That is an oracle bound, not
a result -- it is here to answer whether seed 43 is genuinely worse or merely
badly calibrated.

Attach: the TS-SatFire dataset, plus the dataset holding the five FP checkpoints
and `fp_stats.npz`.


In [1]:
import os, glob, json, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")
try:
    import rasterio
    HAS_RASTERIO = True
except Exception:
    import tifffile
    HAS_RASTERIO = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)
print("torch", torch.__version__, "| device", DEVICE)


class CFG:
    CROP = 256
    TS = 2
    FP_CHANNELS = 27
    FP_BAND_IDX = [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
    SAT_MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                         294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    SAT_STD = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                        24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    # Validation-selected threshold from each training run's JSON.
    # 42, 43 and 44 all recorded 0.75. FILL IN 45 AND 46 from their JSONs.
    THRESHOLDS = {42: 0.75, 43: 0.75, 44: 0.75, 45: 0.75, 46: 0.75}

    SWEEP = np.round(np.arange(0.05, 0.96, 0.05), 2)

    # Excluded in the manuscript (16-fire subset)
    EXCLUDE_16 = [
        "US_2021_FL2521008104520210308", "US_2021_MT4714310953420211004",
        "US_2021_NM3323810847220210520", "US_2021_NM3340210587120210426",
        "US_2021_NM3344410803520210514", "US_2021_NM3676810505920211120",
        "US_2021_AZ3345510938920210616", "US_2021_AZ3368910927620210616",
    ]
    # Rule A: fewer than 3 windows containing any progression (18-fire subset)
    EXCLUDE_18 = [
        "US_2021_FL2521008104520210308", "US_2021_NM3340210587120210426",
        "US_2021_NM3676810505920211120", "US_2021_NM3344410803520210514",
        "US_2021_NM3323810847220210520", "US_2021_MT4714310953420211004",
    ]


def find_data_root():
    for p in ["/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire/ts-satfire/ts-satfire",
              "/kaggle/input/ts-satfire/ts-satfire", "/kaggle/input/ts-satfire"]:
        if os.path.isdir(p):
            return p
    return None


DATA_ROOT = find_data_root()
assert DATA_ROOT, "TS-SatFire dataset not found"
print("DATA_ROOT:", DATA_ROOT)

TEST_FIRES = sorted(d for d in os.listdir(DATA_ROOT)
                    if d.startswith("US_2021")
                    and os.path.isdir(os.path.join(DATA_ROOT, d)))
print("official FP test fires:", len(TEST_FIRES))

# ---- locate checkpoints, one per seed ----
ds = os.path.abspath(DATA_ROOT)
found = []
for base in ["/kaggle/input", "/kaggle/working"]:
    if not os.path.isdir(base):
        continue
    for root, dirs, files in os.walk(base):
        if os.path.abspath(root).startswith(ds):
            dirs.clear(); continue
        for f in files:
            if f.endswith((".pt", ".pth")):
                found.append(os.path.join(root, f))

print("\nCheckpoint files found:")
for p in sorted(found):
    print("  ", p)

import re
CKPT = {}
for p in found:
    b = os.path.basename(p).lower()
    if "fp" not in b and "pred" not in b:
        continue
    m = re.search(r"s(\d{2})", b)
    if m and int(m.group(1)) in CFG.THRESHOLDS:
        CKPT[int(m.group(1))] = p
    elif "fp" in b and 42 not in CKPT:
        CKPT[42] = p          # unsuffixed file assumed to be seed 42

SEEDS = sorted(CKPT)
print("\nResolved:")
for s in SEEDS:
    print(f"  seed {s}: {os.path.basename(CKPT[s])}  thr {CFG.THRESHOLDS[s]}")
assert SEEDS, "No FP checkpoints resolved. Edit CKPT manually."

FP_STATS = None
for pat in ["/kaggle/input/**/fp_stats*.npz", "/kaggle/working/**/fp_stats*.npz"]:
    h = sorted(glob.glob(pat, recursive=True))
    if h:
        FP_STATS = h[0]; break
assert FP_STATS, "fp_stats*.npz not found"
_z = np.load(FP_STATS)
FP_MEAN, FP_STD = _z["mean"].astype(np.float32), _z["std"].astype(np.float32)
print("\nFP stats:", FP_STATS, "| shapes", FP_MEAN.shape, FP_STD.shape)


torch 2.10.0+cu128 | device cuda
DATA_ROOT: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire
official FP test fires: 24

Checkpoint files found:
   /kaggle/input/datasets/nmavros/af-all-weights/best_fp_v6_s43.pth
   /kaggle/input/datasets/nmavros/af-all-weights/best_fp_v6_s44.pth
   /kaggle/input/datasets/nmavros/af-all-weights/fp_v6_best.pth

Resolved:
  seed 42: fp_v6_best.pth  thr 0.75
  seed 43: best_fp_v6_s43.pth  thr 0.75
  seed 44: best_fp_v6_s44.pth  thr 0.75

FP stats: /kaggle/input/datasets/nmavros/af-all-weights/fp_stats.npz | shapes (18,) (18,)


In [2]:
class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8, min_mid=False):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        mid = max(ch // r, 4) if min_mid else ch // r
        self.fc = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(True),
                                nn.Linear(mid, ch, bias=False), nn.Sigmoid())

    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)


class ResBlockFP(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.conv1 = nn.Conv3d(ic, oc, (1, 3, 3), padding=(0, 1, 1), bias=False)
        self.bn1 = nn.BatchNorm3d(oc)
        self.conv2 = nn.Conv3d(oc, oc, (1, 3, 3), padding=(0, 1, 1), bias=False)
        self.bn2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc, 8, min_mid=True)
        self.skip = nn.Conv3d(ic, oc, 1, bias=False) if ic != oc else nn.Identity()
        self.act = nn.GELU()

    def forward(self, x):
        r = self.skip(x)
        o = self.act(self.bn1(self.conv1(x)))
        o = self.bn2(self.conv2(o))
        return self.act(self.se(o) + r)


class ASPP3D(nn.Module):
    def __init__(self, ic, oc, dils=(1, 6, 12)):
        super().__init__()
        bc = oc // len(dils)
        self.br = nn.ModuleList([
            nn.Sequential(nn.Conv3d(ic, bc, (1, 3, 3), padding=(0, d, d),
                                    dilation=(1, d, d), bias=False),
                          nn.BatchNorm3d(bc), nn.GELU())
            for d in dils])
        self.gp = nn.Sequential(nn.AdaptiveAvgPool3d((None, 1, 1)),
                                nn.Conv3d(ic, bc, 1, bias=False),
                                nn.BatchNorm3d(bc), nn.GELU())
        self.fuse = nn.Sequential(nn.Conv3d(bc * (len(dils) + 1), oc, 1, bias=False),
                                  nn.BatchNorm3d(oc), nn.GELU())

    def forward(self, x):
        parts = [b(x) for b in self.br]
        g = self.gp(x).expand(-1, -1, x.shape[2], x.shape[3], x.shape[4])
        parts.append(g)
        return self.fuse(torch.cat(parts, 1))


class SEUNet3DPred(nn.Module):
    """FP variant."""

    def __init__(self, in_ch=27, enc=(64, 128, 256, 512), bneck=1024):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv3d(in_ch, enc[0], (1, 3, 3), padding=(0, 1, 1), bias=False),
            nn.BatchNorm3d(enc[0]), nn.GELU())
        self.encs = nn.ModuleList(); self.pools = nn.ModuleList()
        prev = enc[0]
        for c in enc:
            self.encs.append(ResBlockFP(prev, c))
            self.pools.append(nn.MaxPool3d((1, 2, 2)))
            prev = c
        self.bneck = ASPP3D(enc[-1], bneck)
        self.ups = nn.ModuleList(); self.decs = nn.ModuleList()
        prev = bneck
        for c in reversed(enc):
            self.ups.append(nn.ConvTranspose3d(prev, c, (1, 2, 2), stride=(1, 2, 2)))
            self.decs.append(ResBlockFP(c * 2, c))
            prev = c
        self.head = nn.Sequential(
            nn.Conv2d(enc[0], 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 1, 1))

    def forward(self, x):
        x = self.inp(x)
        skips = []
        for enc, pool in zip(self.encs, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bneck(x)
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            x = up(x)
            d = [sk.shape[i] - x.shape[i] for i in range(2, 5)]
            if any(di != 0 for di in d):
                x = F.pad(x, [0, d[2], 0, d[1], 0, d[0]])
            x = dec(torch.cat([x, sk], 1))
        x = x.mean(dim=2)
        return self.head(x)





In [3]:
# ---------------------------------------------------------------
# Data pipeline. These match the FP training notebook exactly:
#   BA mask   = isfinite(band 8)        (NaN means unburned, not missing)
#   label     = clip(BA(T+1) - BA(T), 0, 1)
#   FirePred  = (x - mean) / max(std, 0.1), clipped to [-5, 5]
#   BA channel= per-day mask, not accumulated across the window
# ---------------------------------------------------------------

def read_tif(path):
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)
    a = tifffile.imread(path).astype(np.float32)
    return a[np.newaxis] if a.ndim == 2 else a


def center_crop(a, size):
    h, w = a.shape[-2], a.shape[-1]
    r0, c0 = max((h - size) // 2, 0), max((w - size) // 2, 0)
    return a[..., r0:r0 + size, c0:c0 + size]


def day_files(fire_dir):
    return sorted(glob.glob(os.path.join(fire_dir, "VIIRS_Day", "*.tif")))


def load_sat8(fire_dir, dpath):
    d = read_tif(dpath)
    bands = d[:6]
    H, W = bands.shape[1], bands.shape[2]
    npath = os.path.join(fire_dir, "VIIRS_Night",
                         os.path.basename(dpath).replace("_VIIRS_Day", "_VIIRS_Night"))
    if os.path.exists(npath):
        n = read_tif(npath)
        nb = n[:2, :H, :W] if n.shape[0] >= 2 else np.zeros((2, H, W), np.float32)
    else:
        nb = np.zeros((2, H, W), np.float32)
    return np.concatenate([bands, nb], axis=0), d


def ba_mask(raw):
    if raw.shape[0] < 8:
        return None
    return np.isfinite(raw[7]).astype(np.float32)


def ba_labelled(raw):
    return raw.shape[0] >= 8 and bool(np.isfinite(raw[7]).any())


def norm_sat(s):
    out = (s - CFG.SAT_MEAN[:, None, None]) / (CFG.SAT_STD[:, None, None] + 1e-8)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)


def norm_fp(a):
    m = FP_MEAN.reshape(-1, 1, 1)
    s = np.maximum(FP_STD, 0.1).reshape(-1, 1, 1)
    return np.nan_to_num(np.clip((a - m) / s, -5.0, 5.0), nan=0.0, posinf=0.0, neginf=0.0)


def build_window(fire_dir, dpaths, raws):
    cs = CFG.CROP
    n_fp = len(CFG.FP_BAND_IDX)
    frames = []
    for t, dp in enumerate(dpaths):
        s, _ = load_sat8(fire_dir, dp)
        if s.shape[1] < cs or s.shape[2] < cs:
            return None
        sat = center_crop(norm_sat(s), cs)

        fpath = os.path.join(fire_dir, "FirePred",
                             os.path.basename(dp).replace("_VIIRS_Day", "_FirePred"))
        fp = np.zeros((n_fp, cs, cs), np.float32)
        if os.path.exists(fpath):
            a = read_tif(fpath)
            if a.shape[0] >= 19:
                a = np.nan_to_num(a[CFG.FP_BAND_IDX], nan=0.0, posinf=0.0, neginf=0.0)
                a = norm_fp(a)
                if a.shape[1] >= cs and a.shape[2] >= cs:
                    fp = center_crop(a, cs)

        b = ba_mask(raws[t])
        ba = center_crop(b, cs)[np.newaxis] if b is not None else np.zeros((1, cs, cs), np.float32)
        frames.append(np.concatenate([sat, fp, ba], axis=0))
    return np.stack(frames, axis=1)      # (27, T, cs, cs)


print("Data pipeline ready.")


Data pipeline ready.


In [4]:
models = {}
for s in SEEDS:
    m = SEUNet3DPred(CFG.FP_CHANNELS).to(DEVICE)
    ck = torch.load(CKPT[s], map_location=DEVICE, weights_only=False)
    st = ck.get("model_state_dict", ck.get("state_dict", ck))
    if any(k.startswith("module.") for k in st):
        st = {k.replace("module.", "", 1): v for k, v in st.items()}
    miss, unexp = m.load_state_dict(st, strict=False)
    m.eval()
    models[s] = m
    flag = "" if not (miss or unexp) else f"  WARNING missing={len(miss)} unexpected={len(unexp)}"
    print(f"seed {s}: loaded {sum(p.numel() for p in m.parameters()):,} params{flag}")

# counts[seed][thr_index] = [tp, fp, fn]; per fire and overall
NT = len(CFG.SWEEP)
per_fire = {s: {} for s in SEEDS}

t0 = time.time()
for k, fid in enumerate(TEST_FIRES, 1):
    fdir = os.path.join(DATA_ROOT, fid)
    files = day_files(fdir)
    acc = {s: np.zeros((NT, 3), dtype=np.int64) for s in SEEDS}
    n_win = 0

    for t0i in range(max(len(files) - CFG.TS, 0)):
        raws, H, W = [], None, None
        ok = True
        for t in range(t0i, t0i + CFG.TS):
            s8, raw = load_sat8(fdir, files[t])
            if H is None:
                H, W = s8.shape[1], s8.shape[2]
            raws.append(raw[:, :H, :W])
        nxt = read_tif(files[t0i + CFG.TS])[:, :H, :W]

        if not ba_labelled(raws[-1]) or not ba_labelled(nxt):
            continue
        bt, bn = ba_mask(raws[-1]), ba_mask(nxt)
        if bt is None or bn is None:
            continue
        label = np.clip(center_crop(bn, CFG.CROP) - center_crop(bt, CFG.CROP), 0, 1)

        x = build_window(fdir, files[t0i:t0i + CFG.TS], raws)
        if x is None:
            continue
        n_win += 1
        xt = torch.from_numpy(x).float().unsqueeze(0).to(DEVICE)
        lab = torch.from_numpy(label).bool().to(DEVICE)

        with torch.no_grad():
            for s in SEEDS:
                with torch.autocast("cuda", enabled=torch.cuda.is_available()):
                    logit = models[s](xt)
                prob = torch.sigmoid(logit.float())[0, 0]
                for ti, thr in enumerate(CFG.SWEEP):
                    p = prob > float(thr)
                    acc[s][ti, 0] += int((p & lab).sum())
                    acc[s][ti, 1] += int((p & ~lab).sum())
                    acc[s][ti, 2] += int((~p & lab).sum())

    for s in SEEDS:
        per_fire[s][fid] = {"n_windows": n_win, "counts": acc[s]}
    print(f"[{k:>2}/{len(TEST_FIRES)}] {fid[8:]:<28} windows {n_win:>3}  "
          f"({(time.time()-t0)/60:.1f} min elapsed)", flush=True)

print(f"\nDone in {(time.time()-t0)/60:.1f} min")


seed 42: loaded 24,279,561 params
seed 43: loaded 24,279,561 params
seed 44: loaded 24,279,561 params
[ 1/24] AZ3345510938920210616        windows  14  (0.4 min elapsed)
[ 2/24] AZ3368910927620210616        windows  16  (0.8 min elapsed)
[ 3/24] CA3451712013120211011        windows   5  (1.0 min elapsed)
[ 4/24] CA3568711855020210818        windows  18  (1.4 min elapsed)
[ 5/24] CA3604711863120210910        windows  22  (2.0 min elapsed)
[ 6/24] CA3627811855020210815        windows  14  (2.3 min elapsed)
[ 7/24] CA3658211879520210912        windows  47  (3.5 min elapsed)
[ 8/24] CA4086312235520210630        windows   8  (3.7 min elapsed)
[ 9/24] FL2521008104520210308        windows   0  (3.8 min elapsed)
[10/24] ID4453211532920210810        windows  44  (5.0 min elapsed)
[11/24] ID4558511544420210705        windows  53  (6.3 min elapsed)
[12/24] ID4663811466720210707        windows  21  (6.8 min elapsed)
[13/24] ID4762711608320210708        windows  56  (8.1 min elapsed)
[14/24] MT4568

In [5]:
def f1_iou(c):
    tp, fp, fn = int(c[0]), int(c[1]), int(c[2])
    f1 = 2 * tp / max(2 * tp + fp + fn, 1)
    iou = tp / max(tp + fp + fn, 1)
    return f1, iou


SUBSETS = {
    "all_24": TEST_FIRES,
    "rule_a_18": [f for f in TEST_FIRES if f not in CFG.EXCLUDE_18],
    "manuscript_16": [f for f in TEST_FIRES if f not in CFG.EXCLUDE_16],
}

rows = []
for s in SEEDS:
    ti_val = int(np.argmin(np.abs(CFG.SWEEP - CFG.THRESHOLDS[s])))
    for name, fires in SUBSETS.items():
        tot = np.zeros(3, dtype=np.int64)
        for f in fires:
            tot += per_fire[s][f]["counts"][ti_val]
        f1, iou = f1_iou(tot)
        # oracle over the sweep on this subset
        sweep_tot = np.zeros((NT, 3), dtype=np.int64)
        for f in fires:
            sweep_tot += per_fire[s][f]["counts"]
        f1s = [f1_iou(sweep_tot[i])[0] for i in range(NT)]
        bi = int(np.argmax(f1s))
        rows.append({"seed": s, "subset": name, "n_fires": len(fires),
                     "val_thr": CFG.THRESHOLDS[s], "f1": f1, "iou": iou,
                     "oracle_thr": float(CFG.SWEEP[bi]), "oracle_f1": f1s[bi]})

df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT, "fp_rescore_per_seed.csv"), index=False)

print("=" * 88)
print("FIRE PROGRESSION -- all seeds, all test-set definitions")
print("=" * 88)
for name in SUBSETS:
    d = df[df.subset == name]
    print(f"\n{name}  (n = {d.n_fires.iloc[0]} fires)")
    print(f"  {'seed':>5} {'thr':>6} {'F1':>9} {'IoU':>9} {'oracle thr':>11} {'oracle F1':>10}")
    for _, r in d.iterrows():
        print(f"  {int(r.seed):>5} {r.val_thr:>6.2f} {r.f1:>9.4f} {r.iou:>9.4f} "
              f"{r.oracle_thr:>11.2f} {r.oracle_f1:>10.4f}")
    print(f"  {'MEAN':>5} {'':>6} {d.f1.mean():>9.4f} {d.iou.mean():>9.4f}")
    print(f"  {'SD':>5} {'':>6} {d.f1.std(ddof=1):>9.4f} {d.iou.std(ddof=1):>9.4f}")

print("\n" + "=" * 88)
print("HEADLINE for the paper (all 24 official test fires):")
d24 = df[df.subset == "all_24"]
print(f"  F1  = {d24.f1.mean():.4f} +/- {d24.f1.std(ddof=1):.4f}   (n = {len(d24)} seeds)")
print(f"  IoU = {d24.iou.mean():.4f} +/- {d24.iou.std(ddof=1):.4f}")

print("\nIs seed 43 genuinely worse, or just badly calibrated?")
d = df[(df.subset == "manuscript_16")].set_index("seed")
if 43 in d.index:
    print(f"  seed 43 at its validation threshold: {d.loc[43,'f1']:.4f}")
    print(f"  seed 43 at its best test threshold:  {d.loc[43,'oracle_f1']:.4f}")
    others = d.drop(43)
    print(f"  other seeds, mean at validation thr: {others.f1.mean():.4f}")
    if d.loc[43, "oracle_f1"] >= others.f1.mean() - 0.01:
        print("  -> calibration. At a better threshold seed 43 matches the others.")
    else:
        print("  -> genuinely a worse solution, not a threshold artefact.")


FIRE PROGRESSION -- all seeds, all test-set definitions

all_24  (n = 24 fires)
   seed    thr        F1       IoU  oracle thr  oracle F1
     42   0.75    0.4129    0.2602        0.70     0.4131
     43   0.75    0.3396    0.2045        0.70     0.3397
     44   0.75    0.4003    0.2503        0.85     0.4030
   MEAN           0.3843    0.2383
     SD           0.0392    0.0297

rule_a_18  (n = 18 fires)
   seed    thr        F1       IoU  oracle thr  oracle F1
     42   0.75    0.4153    0.2621        0.70     0.4156
     43   0.75    0.3433    0.2073        0.70     0.3435
     44   0.75    0.4051    0.2540        0.85     0.4075
   MEAN           0.3879    0.2411
     SD           0.0389    0.0296

manuscript_16  (n = 16 fires)
   seed    thr        F1       IoU  oracle thr  oracle F1
     42   0.75    0.4300    0.2739        0.70     0.4305
     43   0.75    0.3558    0.2164        0.65     0.3563
     44   0.75    0.4249    0.2697        0.80     0.4257
   MEAN           0.4036  

In [6]:
pf = []
for s in SEEDS:
    ti = int(np.argmin(np.abs(CFG.SWEEP - CFG.THRESHOLDS[s])))
    for fid, d in per_fire[s].items():
        f1, iou = f1_iou(d["counts"][ti])
        c = d["counts"][ti]
        pf.append({"seed": s, "fire": fid, "n_windows": d["n_windows"],
                   "tp": int(c[0]), "fp": int(c[1]), "fn": int(c[2]),
                   "f1": f1, "iou": iou})
pd.DataFrame(pf).to_csv(os.path.join(OUT, "fp_rescore_per_fire.csv"), index=False)

summary = {
    "seeds": SEEDS,
    "thresholds": {str(s): CFG.THRESHOLDS[s] for s in SEEDS},
    "subsets": {},
}
for name in SUBSETS:
    d = df[df.subset == name]
    summary["subsets"][name] = {
        "n_fires": int(d.n_fires.iloc[0]),
        "f1_mean": float(d.f1.mean()), "f1_sd": float(d.f1.std(ddof=1)),
        "iou_mean": float(d.iou.mean()), "iou_sd": float(d.iou.std(ddof=1)),
        "per_seed_f1": {str(int(r.seed)): float(r.f1) for _, r in d.iterrows()},
    }
json.dump(summary, open(os.path.join(OUT, "fp_rescore_summary.json"), "w"), indent=2)
print(json.dumps(summary["subsets"], indent=2))
print("\nSend me fp_rescore_summary.json")


{
  "all_24": {
    "n_fires": 24,
    "f1_mean": 0.3842696284651994,
    "f1_sd": 0.039209947735457634,
    "iou_mean": 0.23831053040736272,
    "iou_sd": 0.02968443924504677,
    "per_seed_f1": {
      "42": 0.412899179428213,
      "43": 0.3395791783383539,
      "44": 0.4003305276290313
    }
  },
  "rule_a_18": {
    "n_fires": 18,
    "f1_mean": 0.3879218238555527,
    "f1_sd": 0.03894285008266567,
    "iou_mean": 0.24111134175594337,
    "iou_sd": 0.029601824852806317,
    "per_seed_f1": {
      "42": 0.41533440678189015,
      "43": 0.34334550978887785,
      "44": 0.40508555499589
    }
  },
  "manuscript_16": {
    "n_fires": 16,
    "f1_mean": 0.4035634283348963,
    "f1_sd": 0.041441683210310276,
    "iou_mean": 0.25334502799079567,
    "iou_sd": 0.0320634810157525,
    "per_seed_f1": {
      "42": 0.4300036569899083,
      "43": 0.3558020511641367,
      "44": 0.4248845768506436
    }
  }
}

Send me fp_rescore_summary.json
